<a href="https://colab.research.google.com/github/Leonardozepeda04/etl-data-pipeline2509832022/blob/main/notebooks/C_clientes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd

In [11]:
#URL csv C_clientes
url_C_clientes = "https://raw.githubusercontent.com/Leonardozepeda04/etl-data-pipeline2509832022/refs/heads/main/data/raw/C_clientes.csv"

In [12]:
C_clientes = pd.read_csv(url_C_clientes)

In [13]:
C_clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 155 entries, 0 to 154
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id_cliente  155 non-null    object
 1   cliente     155 non-null    object
 2   correo      152 non-null    object
 3   ciudad      155 non-null    object
dtypes: object(4)
memory usage: 5.0+ KB


In [14]:
# Limpieza general
def limpiar_dataframe(df):
    df = df.copy()

    # Limpiar nombres de columnas
    df.columns = df.columns.str.strip().str.lower()

    # Eliminar espacios en blanco
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()

    # Reemplazar valores vacíos o erróneos por NaN
    df = df.replace(r'^\s*$', pd.NA, regex=True)
    df = df.replace(['error', 'text_43'], pd.NA)

    # Eliminar duplicados
    df = df.drop_duplicates()

    return df
    C_clientes = limpiar_dataframe(C_clientes)

In [23]:
# Transformaciones importantes

# 1. Normalizar nombres de clientes (capitalizar)
C_clientes['cliente'] = C_clientes['cliente'].astype(str).str.strip().str.capitalize()

# 2. Normalizar los correos (convertir todo a minúsculas)
C_clientes['correo'] = C_clientes['correo'].astype(str).str.strip().str.lower()

# 3. Reemplazar valores de correos inválidos con NaN
C_clientes['correo'] = C_clientes['correo'].replace(["", "nan", "None", "correo_invalido"], pd.NA)

# 4. Convertir columnas a 'category' para optimizar memoria
C_clientes['id_cliente'] = C_clientes['id_cliente'].astype("category")
C_clientes['cliente'] = C_clientes['cliente'].astype("category")
C_clientes['correo'] = C_clientes['correo'].astype("category")
C_clientes['ciudad'] = C_clientes['ciudad'].astype("category")

In [24]:
#Verificar resultados
print(C_clientes.head(80))

   id_cliente                   cliente                   correo        ciudad
0       CL100          Comercial díaz 0   cliente065@empresa.com     Santa Ana
1       CL101             Grupo ideal 1     cliente131@correo.sv  San Salvador
2       CL102             Grupo ideal 2   cliente211@empresa.com    San Miguel
3       CL103         Almacenes prado 3     cliente329@gmail.com     Santa Ana
4       CL104           Soluciones cr 4     cliente441@gmail.com   La Libertad
..        ...                       ...                      ...           ...
75      CL175  Importadora del valle 75    cliente7536@gmail.com   La Libertad
76      CL176    Distribuciones luna 76                      NaN     Santa Ana
77      CL177         Tienda central 77  cliente7747@outlook.com    San Miguel
78      CL178         Tienda central 78   cliente784@outlook.com   La Libertad
79      CL179         Servicios alfa 79    cliente7978@correo.sv     Santa Ana

[80 rows x 4 columns]


In [25]:
# Separar válidos y rechazados

# Válidos: Registros donde 'id_cliente', 'cliente' y 'correo' no son nulos ni tienen valores erróneos
validos = C_clientes[
    C_clientes['id_cliente'].notna() &
    C_clientes['cliente'].notna() &
    C_clientes['correo'].notna() &
    ~C_clientes['correo'].isin(['correo_invalido'])  # Excluir correos inválidos
].copy()

# Rechazados: Registros con valores nulos en alguna columna o correo inválido
rechazados = C_clientes[
    C_clientes['id_cliente'].isna() |
    C_clientes['cliente'].isna() |
    C_clientes['correo'].isna() |
    C_clientes['correo'].isin(['correo_invalido'])  # Incluir correos inválidos
].copy()


In [26]:
# Ver los primeros registros de válidos y rechazados
print("\n✅ Válidos:")
print(validos.head())

print("\n❌ Rechazados con motivos:")
print(rechazados[['id_cliente', 'cliente', 'correo']].head())


✅ Válidos:
  id_cliente            cliente                  correo        ciudad
0      CL100   Comercial díaz 0  cliente065@empresa.com     Santa Ana
1      CL101      Grupo ideal 1    cliente131@correo.sv  San Salvador
2      CL102      Grupo ideal 2  cliente211@empresa.com    San Miguel
3      CL103  Almacenes prado 3    cliente329@gmail.com     Santa Ana
4      CL104    Soluciones cr 4    cliente441@gmail.com   La Libertad

❌ Rechazados con motivos:
   id_cliente                 cliente correo
15      CL115      Almacenes prado 15    NaN
36      CL136        Soluciones cr 36    NaN
52      CL152          Grupo ideal 52    NaN
76      CL176  Distribuciones luna 76    NaN
80      CL180       Servicios alfa 80    NaN


In [27]:
# Motivos de rechazo
def motivo_rechazo(row):
    motivos = []

    if pd.isna(row['id_cliente']):
        motivos.append("id_cliente_vacio")
    if pd.isna(row['cliente']):
        motivos.append("cliente_vacio")
    if pd.isna(row['correo']):
        motivos.append("correo_vacio")
    elif row['correo'] == 'correo_invalido':
        motivos.append("correo_invalido")

    return ",".join(motivos)

# Aplicar la función para obtener los motivos de rechazo en los registros rechazados
rechazados["motivo_rechazo"] = rechazados.apply(motivo_rechazo, axis=1)

In [28]:
# Ver los primeros registros de rechazados con sus motivos
print(rechazados[['id_cliente', 'cliente', 'correo', 'motivo_rechazo']].head())

   id_cliente                 cliente correo motivo_rechazo
15      CL115      Almacenes prado 15    NaN   correo_vacio
36      CL136        Soluciones cr 36    NaN   correo_vacio
52      CL152          Grupo ideal 52    NaN   correo_vacio
76      CL176  Distribuciones luna 76    NaN   correo_vacio
80      CL180       Servicios alfa 80    NaN   correo_vacio


In [29]:
# Exportar archivo de datos curados (validos)
validos.to_csv("clientes_validos.csv", index=False)

# Exportar archivo de registros rechazados (rechazados)
rechazados.to_csv("clientes_rechazados.csv", index=False)

# Si también deseas exportar el archivo completo con los datos procesados (curated)
C_clientes.to_csv("clientes_curated.csv", index=False)